# **Boids Part 2 - Pygame, Visualization, Project 2**

In this lab, we'll build a simple predator-prey simulation using Pygame, focusing on both the implementation of agent behaviors and good visualization practices. This exercise will help you understand how complex interactions can emerge from simple rules and how to effectively visualize these interactions.

Learning Objectives:

* Implement basic predator and prey behaviors.
* Understand agent interactions and state transitions.
* Apply good visualization practices to enhance the simulation's clarity and aesthetics.
* Prepare for your graded project by extending this simulation or implementing an epidemic model.

Remember to install pygame on your system.

`pip install pygame`


### Initializing pygame window

In [ ]:
import pygame
import random
import math

# Screen dimensions
WIDTH, HEIGHT = 800, 600

# Colors
BACKGROUND_COLOR = (30, 30, 30)
PREY_COLOR = (0, 255, 0)
PREDATOR_COLOR = (255, 0, 0)


pygame.init()
screen = pygame.display.set_mode((WIDTH, HEIGHT))
pygame.display.set_caption("Predator-Prey Simulation")
clock = pygame.time.Clock()


### Prey Class

In [ ]:
class Prey:
    def __init__(self):
        self.position = pygame.math.Vector2(random.uniform(0, WIDTH), random.uniform(0, HEIGHT))
        angle = random.uniform(0, 2 * math.pi)
        self.velocity = pygame.math.Vector2(math.cos(angle), math.sin(angle))
        self.speed = 2

    def update(self):
        # Move the prey
        self.position += self.velocity * self.speed

        # Bounce off the edges
        if self.position.x < 0 or self.position.x > WIDTH:
            self.velocity.x *= -1
        if self.position.y < 0 or self.position.y > HEIGHT:
            self.velocity.y *= -1

        # Keep position within bounds
        self.position.x = max(0, min(self.position.x, WIDTH))
        self.position.y = max(0, min(self.position.y, HEIGHT))

    def draw(self):
        pygame.draw.circle(screen, PREY_COLOR, (int(self.position.x), int(self.position.y)), 4)

### Predator Class

In [ ]:
class Predator:
    def __init__(self):
        self.position = pygame.math.Vector2(random.uniform(0, WIDTH), random.uniform(0, HEIGHT))
        angle = random.uniform(0, 2 * math.pi)
        self.velocity = pygame.math.Vector2(math.cos(angle), math.sin(angle))
        self.speed = 3

    def update(self, prey_list):
        # Simple hunting behavior: move towards the nearest prey
        if prey_list:
            nearest_prey = min(prey_list, key=lambda prey: self.position.distance_to(prey.position))
            direction = (nearest_prey.position - self.position).normalize()
            self.velocity = direction
            self.position += self.velocity * self.speed
        else:
            # Random movement if no prey
            self.position += self.velocity * self.speed

        # Bounce off the edges
        if self.position.x < 0 or self.position.x > WIDTH:
            self.velocity.x *= -1
        if self.position.y < 0 or self.position.y > HEIGHT:
            self.velocity.y *= -1

        # Keep position within bounds
        self.position.x = max(0, min(self.position.x, WIDTH))
        self.position.y = max(0, min(self.position.y, HEIGHT))

    def draw(self):
        pygame.draw.circle(screen, PREDATOR_COLOR, (int(self.position.x), int(self.position.y)), 6)

### Main Simulation Loop

In [ ]:
# Create lists of prey and predators
prey_list = [Prey() for _ in range(50)]
predator_list = [Predator() for _ in range(3)]

running = True
while running:
    clock.tick(60)  # Limit to 60 FPS
    screen.fill(BACKGROUND_COLOR)

    # Event handling
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            running = False

    # Update and draw prey
    for prey in prey_list[:]:
        prey.update()
        prey.draw()

    # Update and draw predators
    for predator in predator_list:
        predator.update(prey_list)
        predator.draw()

    # Collision detection
    for predator in predator_list:
        for prey in prey_list[:]:
            if predator.position.distance_to(prey.position) < 6:
                prey_list.remove(prey)

    # Update the display
    pygame.display.flip()

pygame.quit()

In this basic simulation:

* Prey move randomly around the screen.

* Predators chase the nearest prey.

* When a predator gets close enough to a prey, the prey is removed from the simulation.

### Enhancing Prey Behavior - Fleeing

In [ ]:
class Prey:
    def __init__(self):
        self.position = pygame.math.Vector2(random.uniform(0, WIDTH), random.uniform(0, HEIGHT))
        self.velocity = pygame.math.Vector2(random.uniform(-1, 1), random.uniform(-1, 1)).normalize()
        self.speed = 2
        self.vision_radius = 50  # New attribute

    def update(self, predators):
        # Flee from predators
        nearest_predator = None
        min_distance = self.vision_radius  # Only consider predators within vision radius

        for predator in predators:
            distance = self.position.distance_to(predator.position)
            if distance < min_distance:
                min_distance = distance
                nearest_predator = predator

        if nearest_predator:
            # Flee from the nearest predator
            flee_direction = (self.position - nearest_predator.position).normalize()
            self.velocity = flee_direction

        # Move the prey
        self.position += self.velocity * self.speed

        # Bounce off the edges
        if self.position.x < 0 or self.position.x > WIDTH:
            self.velocity.x *= -1
        if self.position.y < 0 or self.position.y > HEIGHT:
            self.velocity.y *= -1

        # Keep position within bounds
        self.position.x = max(0, min(self.position.x, WIDTH))
        self.position.y = max(0, min(self.position.y, HEIGHT))

    def draw(self):
        pygame.draw.circle(screen, PREY_COLOR, (int(self.position.x), int(self.position.y)), 4)


# Modify the prey update call in the main loop to pass the list of predators:

# Update and draw prey
for prey in prey_list[:]:
    prey.update(predator_list)
    prey.draw()

# Good Visualization Practices

Now let's focus on improving the visualization of our simulation.

Differentiate predators and prey using shapes and sizes.

* Prey: Small circles.
* Predators: Triangles pointing in the direction of movement.

In [ ]:
class Predator:
    # ... (existing code)
    def draw(self):
        # Calculate the angle in degrees between the velocity and the x-axis
        angle = self.velocity.angle_to(pygame.math.Vector2(1, 0))

        # Define the triangle points relative to the origin, pointing right
        point_list = [
            pygame.math.Vector2(10, 0),    # Tip of the triangle
            pygame.math.Vector2(-5, -5),   # Bottom left
            pygame.math.Vector2(-5, 5),    # Bottom right
        ]

        # Rotate the points and translate to the predator's position
        rotated_point_list = [self.position + p.rotate(-angle) for p in point_list]

        # Draw the predator as a triangle
        pygame.draw.polygon(screen, PREDATOR_COLOR, rotated_point_list)

### Color Coding and Legends
Add a legend to explain the color coding and symbols used.

In [ ]:
def draw_legend():
    font = pygame.font.SysFont(None, 24)
    prey_text = font.render('Prey', True, PREY_COLOR)
    predator_text = font.render('Predator', True, PREDATOR_COLOR)
    screen.blit(prey_text, (10, 10))
    screen.blit(predator_text, (10, 30))

# Add the legend drawing to the main loop
draw_legend()

### Displaying Statistics
Show the number of prey and predators on the screen.

In [ ]:
def draw_stats():
    font = pygame.font.SysFont(None, 24)
    prey_count_text = font.render(f'Prey Count: {len(prey_list)}', True, (200, 200, 200))
    predator_count_text = font.render(f'Predator Count: {len(predator_list)}', True, (200, 200, 200))
    screen.blit(prey_count_text, (WIDTH - 150, 10))
    screen.blit(predator_count_text, (WIDTH - 150, 30))

# Add the statistics drawing to the main loop
draw_stats()

### Play around with coloring and design

Adjust colors for better contrast and visibility.

In [ ]:
# Adjusted Colors
BACKGROUND_COLOR = (25, 25, 40)
PREY_COLOR = (100, 200, 100)
PREDATOR_COLOR = (200, 100, 100)

### Adding Motion Trails
Add trajectory trails to the agents movement.

In [ ]:
class Prey:
    # ... (existing code)
    def __init__(self):
        # ... (existing code)
        self.trail = []

    def update(self, predators):
        # ... (existing code)
        # Update trail
        self.trail.append(self.position.copy())
        if len(self.trail) > 10:
            self.trail.pop(0)

    def draw(self):
        # Draw trail
        if len(self.trail) > 1:
            pygame.draw.lines(screen, PREY_COLOR, False, [(int(p.x), int(p.y)) for p in self.trail], 1)
        pygame.draw.circle(screen, PREY_COLOR, (int(self.position.x), int(self.position.y)), 4)

class Predator:
    # ... (existing code)
    def __init__(self):
        # ... (existing code)
        self.trail = []

    def update(self, prey_list):
        # ... (existing code)
        # Update trail
        self.trail.append(self.position.copy())
        if len(self.trail) > 10:
            self.trail.pop(0)

    def draw(self):
        # Draw trail
        if len(self.trail) > 1:
            pygame.draw.lines(screen, PREDATOR_COLOR, False, [(int(p.x), int(p.y)) for p in self.trail], 1)
        # ... (existing code for drawing predator)


### Adding interaction controls
Introduce keyboard controls to adjust simulation parameters in real-time.

In [ ]:
# Event handling
for event in pygame.event.get():
    if event.type == pygame.QUIT:
        running = False
    # Add prey with 'P' key
    elif event.type == pygame.KEYDOWN:
        if event.key == pygame.K_p:
            prey_list.append(Prey())
        # Add predator with 'O' key
        elif event.key == pygame.K_o:
            predator_list.append(Predator())

# Update the legend
def draw_legend():
    font = pygame.font.SysFont(None, 24)
    prey_text = font.render('Prey (Green Circle) - Press P to add', True, PREY_COLOR)
    predator_text = font.render('Predator (Red Triangle) - Press O to add', True, PREDATOR_COLOR)
    screen.blit(prey_text, (10, 10))
    screen.blit(predator_text, (10, 30))

At this point, we've built a basic predator-prey simulation with enhanced visualization

# Project 2: Agent-based visual simulation

---
## Option 1: Advanced Predator-Prey Simulation

Extend the predator-prey simulation developed in the lab by incorporating additional features and complexities.

### Requirements:

#### Reproduction Mechanism:
- Implement reproduction for prey when certain conditions are met (e.g. high-energy prey will periodically seek each other out - when they meet they stop for a few timesteps and then a 3rd prey agent gets born).
- Implement similar reproduction for predators.

#### Energy Levels:
- Introduce energy levels for both predators and prey.
- Prey lose energy over time and gain energy by eating food resources.
- Predators lose energy over time and gain energy by consuming prey.
- Agents die when their energy reaches zero.

#### Food Resources:
- Add food resources that prey can consume to gain energy.
- Prey is interested in food only when energy is below a certain threshold.
- Visualize food resources on the screen.

#### Improved AI Behaviors:
- Enhance prey behaviors to include flocking (similar to boids).
- Slightly increase speed of agents based on flock size.

#### Obstacle Avoidance:
- Introduce obstacles in the environment that agents must navigate around.

#### History Tracking:
- Save the history of the environment throughout timesteps.
- Include graphs showing:
  - population changes over time
  - birth rates of prey and predators

#### User Interaction:
- Provide any 3 interactible controls to adjust simulation in real time (e.g., reproduction rate, energy consumption, toggling flocking behavior, manually adding food, manually adding agents, manually adding obstacles, etc.).

Play around with your simulation and simulation parameters until you can reach a relatively stable environment state. A state in which both predators and prey cohabit without any or both of the species dying out completely.

---

## Project Option 2: Epidemic Simulation (SIR Model)

Develop an epidemic simulation modeling the spread of an infectious disease within a population using the SIR (Susceptible, Infected, Recovered) model.

### Requirements:

#### Agent States:
- Implement Susceptible, Infected, and Recovered states for agents.
- Visualize each state with distinct colors.

#### State Transitions:
- Define probabilities for infection upon contact between susceptible and infected agents.
- Infection occurs when agents come in proximity to one another, and probability of infection increases with duration of time spent close to one another.
- Implement recovery after a certain period and with certain probability. If recovery is not successful, remove the agent from simulation (simulate death).

#### Movement Behaviors:
- Agents move within the environment, possibly with social behaviors (e.g., temporary grouping, but not flocking).

#### Quarantine Zones:
- Implement a quarantine mechanism. Areas where infected individuals can be isolated until recovery.

#### Vaccination Strategies:
- Simulate the impact of vaccination by making a percentage of the population immune. Vaccination immunization has a succes rate decided by a probability you set. Vaccination probability (probability that agent decides to get vaccinated) is again something you set.

#### History Tracking:
- Display graphs showing the number of agents in each state over time.
- Have a graph for at least infection rate and recovery rate.

#### User Interaction:
- Provide controls to adjust infection, recovery and vaccination rates.

Simulate two scenarios - one in which all agents disappear (death by non-recovery on infection) and one in which at least some agents survive and the virus disappears.

---

### (Optional) Implement both options for extra points.

In [ ]:
# Below is a cleaner implementation of our basic prey/predator system from the lab
import pygame
import random
import math

# Initialize Pygame
pygame.init()

# Screen dimensions
WIDTH, HEIGHT = 800, 600

# Colors
BACKGROUND_COLOR = (30, 30, 30)
PREY_COLOR = (0, 255, 0)
PREDATOR_COLOR = (255, 0, 0)
TEXT_COLOR = (200, 200, 200)

# Frame rate
FPS = 60

# Initialize screen and clock
screen = pygame.display.set_mode((WIDTH, HEIGHT))
pygame.display.set_caption("Predator-Prey Simulation")
clock = pygame.time.Clock()

# Font for text
FONT = pygame.font.SysFont(None, 24)

class Agent:
    """Base class for all agents in the simulation."""
    def __init__(self, position=None, velocity=None, speed=2, color=PREY_COLOR):
        self.position = position or pygame.math.Vector2(random.uniform(0, WIDTH), random.uniform(0, HEIGHT))
        self.velocity = velocity or pygame.math.Vector2(random.uniform(-1, 1), random.uniform(-1, 1)).normalize()
        self.speed = speed
        self.color = color
        self.trail = []
        self.max_trail_length = 10

    def update_position(self):
        """Update the agent's position based on its velocity and speed."""
        self.position += self.velocity * self.speed
        self._bounce_off_walls()
        self._update_trail()

    def _bounce_off_walls(self):
        """Bounce the agent off the screen edges."""
        if self.position.x < 0 or self.position.x > WIDTH:
            self.velocity.x *= -1
        if self.position.y < 0 or self.position.y > HEIGHT:
            self.velocity.y *= -1

        # Keep position within bounds
        self.position.x = max(0, min(self.position.x, WIDTH))
        self.position.y = max(0, min(self.position.y, HEIGHT))

    def _update_trail(self):
        """Update the trail of the agent for visualization."""
        self.trail.append(self.position.copy())
        if len(self.trail) > self.max_trail_length:
            self.trail.pop(0)

    def draw_trail(self):
        """Draw the trail of the agent."""
        if len(self.trail) > 1:
            pygame.draw.lines(screen, self.color, False, [(int(p.x), int(p.y)) for p in self.trail], 1)

    def draw(self):
        """Method to draw the agent. To be implemented by subclasses."""
        raise NotImplementedError("Draw method must be implemented by subclasses.")

class Prey(Agent):
    """Class representing a prey agent."""
    def __init__(self):
        super().__init__(speed=2, color=PREY_COLOR)
        self.vision_radius = 50  # Detection radius for predators

    def update(self, predators):
        """Update the prey's state based on nearby predators."""
        nearest_predator = self._find_nearest_predator(predators)
        if nearest_predator:
            self.flee_from(nearest_predator)
        self.update_position()

    def _find_nearest_predator(self, predators):
        """Find the nearest predator within vision radius."""
        nearest = None
        min_distance = self.vision_radius
        for predator in predators:
            distance = self.position.distance_to(predator.position)
            if distance < min_distance:
                min_distance = distance
                nearest = predator
        return nearest

    def flee_from(self, predator):
        """Change velocity to flee away from the predator."""
        flee_direction = (self.position - predator.position).normalize()
        self.velocity = flee_direction

    def draw(self):
        """Draw the prey as a circle with its trail."""
        pygame.draw.circle(screen, self.color, (int(self.position.x), int(self.position.y)), 4)
        self.draw_trail()

class Predator(Agent):
    """Class representing a predator agent."""
    def __init__(self):
        super().__init__(speed=3, color=PREDATOR_COLOR)

    def update(self, prey_list):
        """Update the predator's state based on nearby prey."""
        if prey_list:
            nearest_prey = self._find_nearest_prey(prey_list)
            if nearest_prey:
                self.hunt(nearest_prey)
        else:
            self.update_position()

    def _find_nearest_prey(self, prey_list):
        """Find the nearest prey."""
        return min(prey_list, key=lambda prey: self.position.distance_to(prey.position), default=None)

    def hunt(self, prey):
        """Change velocity to move towards the prey."""
        direction = (prey.position - self.position).normalize()
        self.velocity = direction
        self.update_position()

    def draw(self):
        """Draw the predator as a rotated triangle with its trail."""
        # Calculate the angle in degrees between the velocity and the x-axis
        angle = self.velocity.angle_to(pygame.math.Vector2(1, 0))

        # Define the triangle points relative to the origin, pointing right
        point_list = [
            pygame.math.Vector2(10, 0),   # Tip of the triangle
            pygame.math.Vector2(-5, -5),  # Bottom left
            pygame.math.Vector2(-5, 5),   # Top left
        ]

        # Rotate the points and translate to the predator's position
        rotated_points = [self.position + p.rotate(-angle) for p in point_list]

        # Draw the predator as a triangle
        pygame.draw.polygon(screen, self.color, rotated_points)

        # Draw the trail
        self.draw_trail()

class Simulation:
    """Class to manage the entire simulation."""
    def __init__(self, num_prey=50, num_predators=3):
        self.prey_list = [Prey() for _ in range(num_prey)]
        self.predator_list = [Predator() for _ in range(num_predators)]
        self.running = True

    def run(self):
        """Main loop of the simulation."""
        while self.running:
            clock.tick(FPS)
            self.handle_events()
            self.update_agents()
            self.handle_collisions()
            self.render()

        pygame.quit()

    def handle_events(self):
        """Handle user input and events."""
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                self.running = False
            elif event.type == pygame.KEYDOWN:
                if event.key == pygame.K_p:
                    self.add_prey()
                elif event.key == pygame.K_o:
                    self.add_predator()

    def add_prey(self):
        """Add a new prey to the simulation."""
        self.prey_list.append(Prey())

    def add_predator(self):
        """Add a new predator to the simulation."""
        self.predator_list.append(Predator())

    def update_agents(self):
        """Update all agents in the simulation."""
        for prey in self.prey_list:
            prey.update(self.predator_list)

        for predator in self.predator_list:
            predator.update(self.prey_list)

    def handle_collisions(self):
        """Handle collisions between predators and prey."""
        for predator in self.predator_list:
            for prey in self.prey_list[:]:
                if predator.position.distance_to(prey.position) < 6:
                    self.prey_list.remove(prey)

    def render(self):
        """Render all elements on the screen."""
        screen.fill(BACKGROUND_COLOR)
        self.draw_legend()
        self.draw_stats()

        # Draw all prey
        for prey in self.prey_list:
            prey.draw()

        # Draw all predators
        for predator in self.predator_list:
            predator.draw()

        pygame.display.flip()

    def draw_legend(self):
        """Draw the legend on the screen."""
        prey_text = FONT.render('Prey (Green Circle) - Press P to add', True, PREY_COLOR)
        predator_text = FONT.render('Predator (Red Triangle) - Press O to add', True, PREDATOR_COLOR)
        screen.blit(prey_text, (10, 10))
        screen.blit(predator_text, (10, 30))

    def draw_stats(self):
        """Draw the simulation statistics on the screen."""
        prey_count_text = FONT.render(f'Prey Count: {len(self.prey_list)}', True, TEXT_COLOR)
        predator_count_text = FONT.render(f'Predator Count: {len(self.predator_list)}', True, TEXT_COLOR)
        screen.blit(prey_count_text, (WIDTH - 150, 10))
        screen.blit(predator_count_text, (WIDTH - 150, 30))

if __name__ == "__main__":
    simulation = Simulation()
    simulation.run()
    